# Notebook 1: T-Domain Fitting + Spatial TV Regularization

## Problem Statement

All noise-recovery methods tested so far (NNLS, SG denoising, blind NMF, Semi-NMF)
fail at low photon counts. The root cause: working in the **attenuation domain**
(`A = -ln(T)`) amplifies noise catastrophically when `T -> 0`:

- At `n_incident=2`, many pixels have `T_obs = 0` (zero detected photons)
- `A = -ln(0)` = infinity, clipped to `A = -ln(1e-6) = 13.8`
- The true absorption is typically 0.5-5.0, so 13.8 is a massive outlier
- NNLS tries to fit these outliers, corrupting the entire coefficient vector

## Key Insight: Fit in Transmission Domain

Instead of linearizing via `A = -ln(T)` and solving NNLS, we fit directly in
**transmission domain**:

```
minimize_{c >= 0}  sum_e [ (T_obs(e) - exp(-A @ c)) / sigma(e) ]^2
```

When `T_obs = 0`, the residual is just `(0 - exp(-A @ c)) / sigma` -- bounded and
well-behaved. No log, no clipping, no outliers.

The tradeoff: this is a **nonlinear** optimization (because of `exp(-A @ c)`), so it's
slower than NNLS. But with `scipy.optimize.least_squares` and an analytical Jacobian,
each pixel takes ~1ms -- about 65 seconds for a full 256x256 image.

## Experiments

| # | Experiment | Question |
|---|-----------|----------|
| 1a | Per-pixel T-domain fit | Does T-domain avoid the clipping catastrophe? |
| 1b | Post-hoc TV on T-domain results | Does spatial regularization clean up remaining noise? |
| 1c | Iterative alternating (T-domain + TV) | Does warm-starting from TV-smoothed maps improve convergence? |

## Testing Progression (each experiment)

1. **Clean baseline** -- verify method matches NNLS on noise-free data
2. **Representative pixels** -- 3 pixels across noise levels, coefficient tables + spectral overlays
3. **Full image** -- abundance maps, difference maps vs ground truth, MAE/RMSE metrics


In [ ]:
import os
os.environ["TQDM_DISABLE"] = "1"

import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import least_squares
from scipy.optimize import nnls as scipy_nnls
from skimage.restoration import denoise_tv_chambolle

from pleiades.imaging import (
    HyperspectralLoader,
    DataDegrader,
    PhysicsRecovery,
    HyperspectralData,
)
from pleiades.imaging.config import ImagingConfig
from pleiades.utils.logger import configure_logger

configure_logger(console_level="WARNING")

DEBUG_DIR = Path("../../_debug_images")
DEBUG_DIR.mkdir(exist_ok=True)

def save_fig(fig, name):
    fig.savefig(DEBUG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

print("Imports OK")


In [ ]:
# --- Representative pixels (identified from clean NNLS ground truth) ---
REP_PIXELS = {
    "Pure U-235": (155, 47),
    "Pure Pu-241": (112, 174),
    "Overlap": (136, 223),
}


def plot_abundance_maps(abundance_maps, isotope_names, title, success_mask=None,
                        vmin=0, vmax=1):
    """Plot abundance maps for each isotope."""
    n_iso = abundance_maps.shape[0]
    fig, axes = plt.subplots(1, n_iso, figsize=(6 * n_iso, 5))
    if n_iso == 1:
        axes = [axes]
    for i, iso in enumerate(isotope_names):
        amap = abundance_maps[i].copy()
        if success_mask is not None:
            amap[~success_mask] = np.nan
        im = axes[i].imshow(amap, cmap="viridis", origin="upper",
                            vmin=vmin, vmax=vmax)
        axes[i].set_title(iso)
        fig.colorbar(im, ax=axes[i], shrink=0.8)
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    return fig


def plot_difference_maps(abundance_maps, gt_maps, isotope_names, title,
                         success_mask=None):
    """Plot difference (method - ground truth) for each isotope."""
    n_iso = abundance_maps.shape[0]
    fig, axes = plt.subplots(1, n_iso, figsize=(6 * n_iso, 5))
    if n_iso == 1:
        axes = [axes]
    for i, iso in enumerate(isotope_names):
        diff = abundance_maps[i] - gt_maps[i]
        if success_mask is not None:
            diff[~success_mask] = np.nan
        vmax = max(0.3, np.nanmax(np.abs(diff[np.isfinite(diff)])))
        im = axes[i].imshow(diff, cmap="RdBu_r", origin="upper",
                            vmin=-vmax, vmax=vmax)
        axes[i].set_title(f"{iso} difference")
        fig.colorbar(im, ax=axes[i], shrink=0.8)
    fig.suptitle(f"{title} (red=overestimate, blue=underestimate)", y=1.02)
    fig.tight_layout()
    return fig


def plot_spectra_at_pixels(energy, data_cubes, labels, colors=None,
                           title_suffix=""):
    """Plot transmission spectra at 3 representative pixels.

    data_cubes: list of (n_energy, H, W) arrays
    labels: list of string labels
    colors: optional list of colors
    """
    if colors is None:
        default_colors = ["black", "#d62728", "#2ca02c", "#1f77b4",
                          "#ff7f0e", "#9467bd", "#e377c2"]
        colors = [default_colors[i % len(default_colors)]
                  for i in range(len(labels))]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for col, (pix_label, (pr, pc)) in enumerate(REP_PIXELS.items()):
        ax_top = axes[0, col]
        for idx, (cube, lbl) in enumerate(zip(data_cubes, labels)):
            y = cube[:, pr, pc]
            if idx == 0:
                ax_top.plot(energy, y, lw=1.2, color=colors[idx],
                           label=lbl, zorder=10)
            else:
                ax_top.scatter(energy, y, s=1, alpha=0.5,
                              color=colors[idx], label=lbl, zorder=5 - idx)
        ax_top.set_title(f"{pix_label} ({pr},{pc})")
        ax_top.set_ylabel("Transmission")
        ax_top.set_ylim(-0.05, 1.15)
        ax_top.legend(fontsize=7, markerscale=5)

        ax_bot = axes[1, col]
        clean = data_cubes[0][:, pr, pc]
        for idx, (cube, lbl) in enumerate(zip(data_cubes[1:], labels[1:]), 1):
            residual = cube[:, pr, pc] - clean
            ax_bot.scatter(energy, residual, s=1, alpha=0.5,
                          color=colors[idx], label=lbl)
        ax_bot.axhline(0, color="black", lw=0.5, ls="--")
        ax_bot.set_title("Residual vs clean")
        ax_bot.set_ylabel("T(method) - T(clean)")
        ax_bot.set_xlabel("Energy (eV)")
        ax_bot.legend(fontsize=7, markerscale=5)

    fig.suptitle(f"Spectral Comparison{title_suffix}", y=1.02)
    fig.tight_layout()
    return fig


print(f"Helpers loaded. Representative pixels: {REP_PIXELS}")


In [ ]:
# --- T-Domain and NNLS fitting functions ---

def compute_poisson_sigma(T_obs, n_incident):
    """Poisson noise std-dev in transmission domain.

    sigma = sqrt(max(T, 1/n) / n), floored at 1/n (one-count uncertainty).
    """
    T_safe = np.maximum(T_obs, 1.0 / n_incident)
    sigma = np.sqrt(T_safe / n_incident)
    return np.maximum(sigma, 1.0 / n_incident)


def fit_pixel_tdomain(T_obs, sigma, A_matrix, x0=None):
    """Fit one pixel in transmission domain via nonlinear least squares.

    Minimizes: sum_e [(T_obs(e) - exp(-A @ c)) / sigma(e)]^2
    subject to: c >= 0

    Uses scipy.optimize.least_squares with trust-region reflective
    algorithm and analytical Jacobian.

    Returns: (coefficients, cost, success)
    """
    n_iso = A_matrix.shape[1]
    if x0 is None:
        x0 = np.full(n_iso, 0.1)

    sigma_safe = np.maximum(sigma, 1e-10)

    def residual(c):
        T_model = np.exp(-A_matrix @ c)
        return (T_obs - T_model) / sigma_safe

    def jacobian(c):
        T_model = np.exp(-A_matrix @ c)
        # d(residual_e)/d(c_i) = A[e,i] * exp(-A @ c) / sigma[e]
        return (A_matrix * T_model[:, np.newaxis]) / sigma_safe[:, np.newaxis]

    try:
        result = least_squares(
            residual, x0, jac=jacobian, method="trf",
            bounds=(0, np.inf), max_nfev=200,
        )
        return result.x, result.cost, result.success
    except Exception:
        return np.zeros(n_iso), np.nan, False


def fit_image_tdomain(data_cube, sigma_cube, A_matrix, x0_map=None, label=""):
    """Fit all pixels in transmission domain.

    Parameters
    ----------
    data_cube : (n_energy, H, W)
    sigma_cube : (n_energy, H, W)
    A_matrix : (n_energy, n_isotopes)
    x0_map : optional (n_isotopes, H, W) initial guesses
    label : progress label

    Returns
    -------
    coeff_maps : (n_isotopes, H, W)
    cost_map : (H, W)
    success_mask : (H, W) bool
    """
    n_e, H, W = data_cube.shape
    n_iso = A_matrix.shape[1]
    coeff_maps = np.full((n_iso, H, W), np.nan)
    cost_map = np.full((H, W), np.nan)
    success_mask = np.zeros((H, W), dtype=bool)

    t0 = time.time()
    for r in range(H):
        for c in range(W):
            T_obs = data_cube[:, r, c]
            sig = sigma_cube[:, r, c]
            x0 = x0_map[:, r, c] if x0_map is not None else None
            coeffs, cost, ok = fit_pixel_tdomain(T_obs, sig, A_matrix, x0)
            if ok:
                coeff_maps[:, r, c] = coeffs
                cost_map[r, c] = cost
                success_mask[r, c] = True
        if (r + 1) % 64 == 0:
            elapsed = time.time() - t0
            rate = (r + 1) * W / elapsed
            eta = (H - r - 1) * W / rate
            print(f"  {label} row {r+1}/{H} "
                  f"({rate:.0f} px/s, ETA {eta:.0f}s)")

    elapsed = time.time() - t0
    n_ok = int(success_mask.sum())
    print(f"  {label} done: {n_ok}/{H*W} pixels in {elapsed:.1f}s")
    return coeff_maps, cost_map, success_mask


def coeffs_to_abundances(coeff_maps, success_mask):
    """Normalize coefficients to abundances (sum to 1 per pixel)."""
    abundance = coeff_maps.copy()
    total = np.nansum(abundance, axis=0)
    nonzero = (total > 0) & success_mask
    for i in range(abundance.shape[0]):
        abundance[i, nonzero] /= total[nonzero]
    zero = (total == 0) | ~success_mask
    for i in range(abundance.shape[0]):
        abundance[i, zero] = np.nan
    return abundance


def fit_pixel_nnls(T_obs, sigma, A_matrix):
    """Baseline NNLS in attenuation domain (for comparison).

    Converts T_obs -> A_obs = -ln(clip(T, 1e-6, 1-1e-6)),
    then solves weighted NNLS: A_obs = A_matrix @ c, c >= 0.

    Returns: (coefficients, success)
    """
    T_clamp = np.clip(T_obs, 1e-6, 1.0 - 1e-6)
    A_obs = -np.log(T_clamp)
    sigma_abs = np.maximum(sigma, 1e-10) / T_clamp
    weights = 1.0 / np.maximum(sigma_abs, 1e-10)
    try:
        c, _ = scipy_nnls(weights[:, np.newaxis] * A_matrix, weights * A_obs)
        return c, True
    except Exception:
        return np.zeros(A_matrix.shape[1]), False


def fit_image_nnls(data_cube, sigma_cube, A_matrix, label=""):
    """NNLS in attenuation domain for all pixels."""
    n_e, H, W = data_cube.shape
    n_iso = A_matrix.shape[1]
    coeff_maps = np.full((n_iso, H, W), np.nan)
    success_mask = np.zeros((H, W), dtype=bool)
    t0 = time.time()
    for r in range(H):
        for c in range(W):
            coeffs, ok = fit_pixel_nnls(
                data_cube[:, r, c], sigma_cube[:, r, c], A_matrix)
            if ok:
                coeff_maps[:, r, c] = coeffs
                success_mask[r, c] = True
    elapsed = time.time() - t0
    n_ok = int(success_mask.sum())
    print(f"  {label} NNLS done: {n_ok}/{H*W} in {elapsed:.1f}s")
    return coeff_maps, success_mask


def apply_tv(abundance_maps, weight, success_mask=None):
    """Apply TV denoising to each isotope abundance map.

    NaN pixels are set to 0 before denoising, restored after.
    Non-negativity enforced after TV.
    """
    smoothed = np.empty_like(abundance_maps)
    for i in range(abundance_maps.shape[0]):
        amap = abundance_maps[i].copy()
        nan_mask = np.isnan(amap)
        amap[nan_mask] = 0.0
        smoothed[i] = denoise_tv_chambolle(amap, weight=weight)
        smoothed[i] = np.maximum(smoothed[i], 0.0)
        smoothed[i][nan_mask] = np.nan
    return smoothed


def compute_mae(pred, gt, mask=None):
    """Mean absolute error over valid pixels."""
    if mask is None:
        mask = np.isfinite(pred) & np.isfinite(gt)
    else:
        mask = mask & np.isfinite(pred) & np.isfinite(gt)
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs(pred[mask] - gt[mask])))


def compute_rmse(pred, gt, mask=None):
    """Root mean square error over valid pixels."""
    if mask is None:
        mask = np.isfinite(pred) & np.isfinite(gt)
    else:
        mask = mask & np.isfinite(pred) & np.isfinite(gt)
    if mask.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((pred[mask] - gt[mask]) ** 2)))


print("T-domain fitting functions defined.")


In [ ]:
sammy_executable = Path("/Users/8cz/code-int.ornl.gov/sammy/build/bin/sammy")
tiff_path = Path("../../tests/data/pleiades_data/LANL-ORNL_example.tif")
energy = np.linspace(1.0, 50.0, 500)
isotope_names = ["U-235", "Pu-241"]

config = ImagingConfig(
    isotopes=isotope_names,
    element="U",
    mass_number=235,
    density_g_cm3=19.1,
    thickness_mm=0.21,
    atomic_mass_amu=235.0439,
    natural_abundances=False,
    custom_abundances=[0.5, 0.5],
    min_energy_eV=1.0,
    max_energy_eV=50.0,
    temperature_K=293.6,
    fit_abundances=True,
)

print(f"SAMMY: {sammy_executable}")
print(f"Isotopes: {config.isotopes}")


In [ ]:
# --- Load clean data ---
loader = HyperspectralLoader(tiff_path, energy=energy)
hyperspectral = loader.load()

# Set baseline uncertainty for clean data
hyperspectral.uncertainty = np.maximum(
    hyperspectral.uncertainty,
    0.02 * np.abs(hyperspectral.data) + 1e-4,
).astype(np.float32)

n_energy, height, width = hyperspectral.shape
print(f"Shape: {height}x{width} pixels, {n_energy} energy bins")

# --- Generate SAMMY reference spectra ---
recovery = PhysicsRecovery(imaging_config=config, sammy_executable=sammy_executable)
ref_spectra = recovery.generate_reference_spectra(energy)
print(f"Reference spectra: {[r.isotope_name for r in ref_spectra]}")
for ref in ref_spectra:
    print(f"  {ref.isotope_name}: T range "
          f"[{ref.transmission.min():.4f}, {ref.transmission.max():.4f}]")

# --- Build absorption matrix A (n_energy, n_isotopes) ---
A_matrix = np.column_stack([r.absorption for r in ref_spectra])
print(f"\nA_matrix shape: {A_matrix.shape}")
print(f"A_matrix range: [{A_matrix.min():.4f}, {A_matrix.max():.4f}]")


In [ ]:
# --- Generate noisy datasets at multiple count levels ---
noise_levels = [2, 5, 10, 50, 500]
noisy_data = {}   # n_inc -> (n_energy, H, W)
sigma_data = {}   # n_inc -> (n_energy, H, W)

# Also store a constant sigma for clean data
sigma_clean = 0.01 * np.ones_like(hyperspectral.data)

print(f"{'n_incident':>10} {'Zero-T pixels':>15} {'Max T_obs':>10}")
print("-" * 40)
for n_inc in noise_levels:
    degrader = DataDegrader(random_seed=42)
    data_noisy = degrader.add_poisson_noise(hyperspectral.data, n_incident=n_inc)
    noisy_data[n_inc] = data_noisy
    sigma_data[n_inc] = compute_poisson_sigma(data_noisy, n_inc)

    n_zero = (data_noisy == 0).sum()
    pct_zero = 100.0 * n_zero / data_noisy.size
    print(f"{n_inc:>10} {pct_zero:>14.1f}% {data_noisy.max():>10.3f}")

# Visualize noise at a representative pixel
pix_r, pix_c = REP_PIXELS["Overlap"]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flat
axes[0].plot(energy, hyperspectral.data[:, pix_r, pix_c], lw=0.5)
axes[0].set_title("Clean")
axes[0].set_ylim(-0.1, 1.3)
for idx, n_inc in enumerate(noise_levels):
    ax = axes[idx + 1]
    ax.scatter(energy, noisy_data[n_inc][:, pix_r, pix_c], s=1, alpha=0.6)
    ax.plot(energy, hyperspectral.data[:, pix_r, pix_c], lw=0.8, alpha=0.5,
            color="black")
    ax.set_title(f"n_incident={n_inc}")
    ax.set_ylim(-0.1, 1.3)
for ax in axes:
    ax.set_xlabel("Energy (eV)")
    ax.set_ylabel("Transmission")
fig.suptitle(f"Overlap pixel ({pix_r},{pix_c}): Poisson noise", y=1.02)
fig.tight_layout()
save_fig(fig, "NB1_00_noise_levels")


In [ ]:
# --- Ground truth: NNLS on clean data ---
results_clean = recovery.recover_image(hyperspectral, reference_spectra=ref_spectra)
gt_abundances = results_clean.abundance_maps  # (n_isotopes, H, W)
gt_mask = results_clean.success_mask

print("Ground truth (NNLS on clean data):")
for i, iso in enumerate(isotope_names):
    valid = gt_abundances[i][gt_mask]
    print(f"  {iso}: mean={np.nanmean(valid):.4f}, "
          f"std={np.nanstd(valid):.4f}, "
          f"range=[{np.nanmin(valid):.4f}, {np.nanmax(valid):.4f}]")

fig = plot_abundance_maps(gt_abundances, isotope_names,
                          "Ground Truth: NNLS on Clean Data", gt_mask)
save_fig(fig, "NB1_00_ground_truth")


---

# Experiment 1a: Per-Pixel T-Domain Fitting

**Question**: Does fitting in the transmission domain avoid the clipping catastrophe
that destroys NNLS at low photon counts?

**Method**:

For each pixel, solve:
```
minimize_{c >= 0}  sum_e [ (T_obs(e) - exp(-A @ c)) / sigma(e) ]^2
```
using `scipy.optimize.least_squares(method='trf')` with analytical Jacobian:
```
J[e, i] = A[e,i] * exp(-A @ c) / sigma(e)
```

**Comparison**: Weighted NNLS in attenuation domain (`A = -ln(T)`), using the same
Poisson-derived sigma for fair comparison.

**Key questions**:
1. Does T-domain match NNLS on clean data? (sanity check)
2. At what noise level does T-domain start outperforming NNLS?
3. How does T-domain handle the T=0 (zero-count) bins that destroy NNLS?


In [ ]:
# --- 1a: Clean data validation ---
# Both T-domain and NNLS should produce identical results on noise-free data.

print("=== Clean Data Validation ===\n")

# T-domain on clean data
td_clean, td_clean_cost, td_clean_mask = fit_image_tdomain(
    hyperspectral.data, sigma_clean, A_matrix, label="Clean T-domain")
td_clean_abund = coeffs_to_abundances(td_clean, td_clean_mask)

# NNLS on clean data (already have gt_abundances from cell 7)
print()

# Compare at representative pixels
print(f"{'Pixel':<16} {'Iso':<8} {'NNLS':>8} {'T-domain':>10} {'Diff':>8}")
print("-" * 52)
for pname, (pr, pc) in REP_PIXELS.items():
    for i, iso in enumerate(isotope_names):
        v_nnls = gt_abundances[i, pr, pc]
        v_td = td_clean_abund[i, pr, pc]
        diff = abs(v_td - v_nnls) if np.isfinite(v_td) and np.isfinite(v_nnls) else np.nan
        print(f"{pname:<16} {iso:<8} {v_nnls:>8.4f} {v_td:>10.4f} {diff:>8.6f}")

# Full-image MAE between T-domain and NNLS on clean data
valid = gt_mask & td_clean_mask
for i, iso in enumerate(isotope_names):
    mae = compute_mae(td_clean_abund[i], gt_abundances[i], valid)
    print(f"\n{iso} full-image MAE (T-domain vs NNLS, clean): {mae:.6f}")

# Visual: overlay both at representative pixels
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for col, (pname, (pr, pc)) in enumerate(REP_PIXELS.items()):
    T_obs = hyperspectral.data[:, pr, pc]
    c_td = td_clean[:, pr, pc]
    c_nnls_raw = np.array([gt_abundances[i, pr, pc] for i in range(len(isotope_names))])
    # Reconstruct fitted spectra
    T_fit_td = np.exp(-A_matrix @ c_td)
    axes[col].plot(energy, T_obs, "k-", lw=0.8, label="Clean observed")
    axes[col].plot(energy, T_fit_td, "r--", lw=0.8, label="T-domain fit")
    axes[col].set_title(f"{pname}")
    axes[col].set_xlabel("Energy (eV)")
    axes[col].set_ylabel("Transmission")
    axes[col].legend(fontsize=8)
fig.suptitle("1a: Clean Validation -- T-domain fit vs observed", y=1.02)
fig.tight_layout()
save_fig(fig, "NB1_01a_clean_validation")


In [ ]:
# --- 1a: Representative pixel benchmarks across noise levels ---

import pandas as pd

rows = []
for n_inc in noise_levels:
    T_data = noisy_data[n_inc]
    sig = sigma_data[n_inc]

    for pname, (pr, pc) in REP_PIXELS.items():
        T_obs = T_data[:, pr, pc]
        s = sig[:, pr, pc]

        # T-domain fit
        c_td, cost_td, ok_td = fit_pixel_tdomain(T_obs, s, A_matrix)
        total_td = c_td.sum()
        abund_td = c_td / total_td if total_td > 0 else c_td

        # NNLS fit
        c_nnls, ok_nnls = fit_pixel_nnls(T_obs, s, A_matrix)
        total_nnls = c_nnls.sum()
        abund_nnls = c_nnls / total_nnls if total_nnls > 0 else c_nnls

        # Ground truth abundances
        gt = [gt_abundances[i, pr, pc] for i in range(len(isotope_names))]

        for i, iso in enumerate(isotope_names):
            rows.append({
                "n_incident": n_inc,
                "pixel": pname,
                "isotope": iso,
                "gt": gt[i],
                "nnls_coeff": c_nnls[i],
                "nnls_abund": abund_nnls[i],
                "td_coeff": c_td[i],
                "td_abund": abund_td[i],
                "nnls_err": abs(abund_nnls[i] - gt[i]),
                "td_err": abs(abund_td[i] - gt[i]),
            })

df = pd.DataFrame(rows)

# Print summary table
print("=== Representative Pixel Benchmarks ===")
print(f"{'n':>4} {'Pixel':<14} {'Iso':<7} {'GT':>6} "
      f"{'NNLS':>6} {'err':>6} {'T-dom':>6} {'err':>6} {'Winner':>8}")
print("-" * 72)
for _, r in df.iterrows():
    winner = "T-dom" if r["td_err"] < r["nnls_err"] else "NNLS"
    if abs(r["td_err"] - r["nnls_err"]) < 0.005:
        winner = "tie"
    print(f"{r['n_incident']:>4} {r['pixel']:<14} {r['isotope']:<7} "
          f"{r['gt']:>6.3f} {r['nnls_abund']:>6.3f} {r['nnls_err']:>6.3f} "
          f"{r['td_abund']:>6.3f} {r['td_err']:>6.3f} {winner:>8}")

# Spectral overlay at n=2 (hardest case)
n_inc = 2
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for col, (pname, (pr, pc)) in enumerate(REP_PIXELS.items()):
    T_obs = noisy_data[n_inc][:, pr, pc]
    s = sigma_data[n_inc][:, pr, pc]
    T_clean = hyperspectral.data[:, pr, pc]

    c_td, _, _ = fit_pixel_tdomain(T_obs, s, A_matrix)
    c_nnls, _ = fit_pixel_nnls(T_obs, s, A_matrix)
    T_fit_td = np.exp(-A_matrix @ c_td)
    T_fit_nnls = np.exp(-A_matrix @ c_nnls)

    # Top: spectra
    ax = axes[0, col]
    ax.plot(energy, T_clean, "k-", lw=1, label="Clean truth", zorder=10)
    ax.scatter(energy, T_obs, s=1, alpha=0.4, color="#d62728",
               label=f"Noisy (n={n_inc})", zorder=1)
    ax.plot(energy, T_fit_td, color="#2ca02c", lw=1.2,
            label="T-domain fit", zorder=8)
    ax.plot(energy, T_fit_nnls, color="#1f77b4", lw=1.2, ls="--",
            label="NNLS fit", zorder=7)
    ax.set_title(f"{pname}")
    ax.set_ylim(-0.05, 1.15)
    ax.set_ylabel("Transmission")
    ax.legend(fontsize=7, markerscale=5)

    # Bottom: residual vs clean
    ax = axes[1, col]
    ax.plot(energy, T_fit_td - T_clean, color="#2ca02c", lw=0.8,
            label="T-domain - clean")
    ax.plot(energy, T_fit_nnls - T_clean, color="#1f77b4", lw=0.8,
            ls="--", label="NNLS - clean")
    ax.axhline(0, color="black", lw=0.5, ls="--")
    ax.set_ylabel("Residual")
    ax.set_xlabel("Energy (eV)")
    ax.legend(fontsize=7)

fig.suptitle(f"1a: T-domain vs NNLS at representative pixels (n={n_inc})",
             y=1.02)
fig.tight_layout()
save_fig(fig, "NB1_01a_pixel_benchmarks")


In [ ]:
# --- 1a: Full-image T-domain vs NNLS at n=2 and n=10 ---

# Store results for later comparison
all_results = {}  # key -> abundance maps

for n_inc in [2, 10]:
    print(f"\n{'='*50}")
    print(f"Full-image fits at n_incident={n_inc}")
    print(f"{'='*50}")

    T_data = noisy_data[n_inc]
    sig = sigma_data[n_inc]

    # T-domain
    td_coeffs, td_cost, td_mask = fit_image_tdomain(
        T_data, sig, A_matrix, label=f"n={n_inc} T-domain")
    td_abund = coeffs_to_abundances(td_coeffs, td_mask)
    all_results[f"tdomain_n{n_inc}"] = td_abund
    all_results[f"tdomain_n{n_inc}_mask"] = td_mask
    all_results[f"tdomain_n{n_inc}_coeffs"] = td_coeffs

    # NNLS
    nnls_coeffs, nnls_mask = fit_image_nnls(
        T_data, sig, A_matrix, label=f"n={n_inc}")
    nnls_abund = coeffs_to_abundances(nnls_coeffs, nnls_mask)
    all_results[f"nnls_n{n_inc}"] = nnls_abund
    all_results[f"nnls_n{n_inc}_mask"] = nnls_mask

    # MAE summary
    valid = td_mask & nnls_mask & gt_mask
    print(f"\n  {'Method':<12} {'U-235 MAE':>12} {'Pu-241 MAE':>12}")
    print(f"  {'-'*38}")
    for method, abund in [("NNLS", nnls_abund), ("T-domain", td_abund)]:
        maes = [compute_mae(abund[i], gt_abundances[i], valid)
                for i in range(len(isotope_names))]
        print(f"  {method:<12} {maes[0]:>12.4f} {maes[1]:>12.4f}")

    # Abundance maps: side by side (GT / NNLS / T-domain)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for row_i, iso in enumerate(isotope_names):
        for col_j, (lbl, abund, mask) in enumerate([
            ("Ground Truth", gt_abundances, gt_mask),
            (f"NNLS (n={n_inc})", nnls_abund, nnls_mask),
            (f"T-domain (n={n_inc})", td_abund, td_mask),
        ]):
            amap = abund[row_i].copy()
            amap[~mask] = np.nan
            im = axes[row_i, col_j].imshow(amap, cmap="viridis",
                                            origin="upper", vmin=0, vmax=1)
            axes[row_i, col_j].set_title(f"{iso} -- {lbl}")
            fig.colorbar(im, ax=axes[row_i, col_j], shrink=0.7)
    fig.suptitle(f"1a: Abundance Maps at n_incident={n_inc}", y=1.02)
    fig.tight_layout()
    save_fig(fig, f"NB1_01a_maps_n{n_inc}")

    # Difference maps
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    for row_i, iso in enumerate(isotope_names):
        for col_j, (lbl, abund, mask) in enumerate([
            (f"NNLS (n={n_inc})", nnls_abund, nnls_mask),
            (f"T-domain (n={n_inc})", td_abund, td_mask),
        ]):
            diff = abund[row_i] - gt_abundances[row_i]
            diff[~(mask & gt_mask)] = np.nan
            vmax = max(0.3, np.nanmax(np.abs(diff[np.isfinite(diff)])))
            im = axes[row_i, col_j].imshow(diff, cmap="RdBu_r",
                                            origin="upper",
                                            vmin=-vmax, vmax=vmax)
            axes[row_i, col_j].set_title(f"{iso} -- {lbl} minus GT")
            fig.colorbar(im, ax=axes[row_i, col_j], shrink=0.7)
    fig.suptitle(f"1a: Difference Maps at n_incident={n_inc}", y=1.02)
    fig.tight_layout()
    save_fig(fig, f"NB1_01a_diff_n{n_inc}")

print("\nExperiment 1a complete.")


### Experiment 1a: Observations

1. **Clean validation**: Do T-domain and NNLS agree on clean data?
   - **No — there is a systematic disagreement.** Full-image MAE between T-domain and NNLS
     on noise-free data is 0.036. At representative pixels, differences range from 0.02
     (Overlap) to 0.088 (Pure Pu-241). This is because the two methods solve *different*
     optimization problems: NNLS minimizes error in the attenuation domain (log space),
     while T-domain minimizes error in transmission space. The different weighting of energy
     bins (T-domain up-weights deep resonances, NNLS down-weights them) leads to slightly
     different coefficient estimates even on perfect data. This ~0.04 MAE floor means
     T-domain cannot be strictly better than NNLS at very low noise — confirmed at n=500
     where NNLS (MAE 0.013) beats T-domain (MAE 0.029).

2. **Representative pixels**: At what noise level does T-domain start winning?
   - **Mixed at n=2, clearly wins at n=10+.** At n=2, results are pixel-dependent: T-domain
     wins at Pure U-235 (err 0.14 vs 0.20) but loses at Pure Pu-241 (err 0.53 vs 0.48).
     At n=10, T-domain wins all 3 pixels decisively (errors 0.018-0.041 vs 0.034-0.166).
     At n=50, T-domain wins 2 of 3 pixels. The crossover where T-domain's noise advantage
     overcomes its systematic bias is around n=5-10.

3. **Full image n=10**: Is the improvement visible in the abundance maps?
   - **Yes, clearly.** The T-domain maps at n=10 show noticeably sharper logo features and
     less noisy backgrounds than NNLS. MAE improves from 0.102 (NNLS) to 0.082 (T-domain),
     a 20% reduction. The difference maps confirm: NNLS shows strong systematic bias
     (deep red in U-235 oak leaf region, deep blue in Pu-241 atom region), while T-domain
     shows much weaker, more symmetric residuals.

4. **Full image n=2**: How large is the difference at extreme noise?
   - **Modest: 8.4% improvement.** MAE goes from 0.154 (NNLS) to 0.141 (T-domain). Both
     maps are very noisy — the logos are barely visible in either. The difference maps show
     T-domain has slightly less systematic bias but similar noise amplitude. The clipping
     catastrophe is real (16.3% of bins have T=0) but is not the *dominant* error source
     at n=2. The dominant issue is simply that 2 photons per energy bin carry almost no
     information regardless of fitting domain.

5. **Spectral fits**: Do the T-domain fitted spectra look physically reasonable?
   - **Yes.** The spectral overlay at n=2 shows both T-domain and NNLS fits track the clean
     truth reasonably well despite extreme noise. Both produce smooth model curves that
     capture the major resonance dips. The residual plots show T-domain and NNLS residuals
     are of similar magnitude, with neither showing obviously better spectral fidelity at
     this noise level.

---

# Experiment 1b: Post-hoc TV Regularization

**Question**: Can Total Variation (TV) denoising of the abundance maps further
improve results by exploiting spatial structure?

**Method**: For each isotope's abundance map, apply TV denoising independently:

```
c_smooth = skimage.restoration.denoise_tv_chambolle(c_map, weight=lambda)
```

TV preserves edges (logo boundaries) while smoothing noise within flat regions.
We sweep `weight` over [0.01, 0.02, 0.05, 0.1, 0.2, 0.5] and evaluate MAE vs
ground truth.

**Key questions**:
1. What is the optimal TV weight for T-domain results vs NNLS results?
2. Does TV help T-domain more or less than it helps NNLS?
3. At high TV weight, does over-smoothing blur the logo edges?


In [ ]:
# --- 1b: TV weight sweep ---
tv_weights = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5]

# Sweep on n=2 (hardest case)
n_inc = 2
td_abund_n2 = all_results[f"tdomain_n{n_inc}"]
td_mask_n2 = all_results[f"tdomain_n{n_inc}_mask"]
nnls_abund_n2 = all_results[f"nnls_n{n_inc}"]
nnls_mask_n2 = all_results[f"nnls_n{n_inc}_mask"]

tv_results = {}  # (method, weight) -> (abundance, mae_u, mae_pu)

for method, abund, mask in [("NNLS", nnls_abund_n2, nnls_mask_n2),
                             ("T-domain", td_abund_n2, td_mask_n2)]:
    for w in tv_weights:
        smoothed = apply_tv(abund, w, mask)
        valid = mask & gt_mask
        mae_u = compute_mae(smoothed[0], gt_abundances[0], valid)
        mae_pu = compute_mae(smoothed[1], gt_abundances[1], valid)
        tv_results[(method, w)] = (smoothed, mae_u, mae_pu)
        all_results[f"{method.lower().replace('-','')}_tv{w}_n{n_inc}"] = smoothed

# Also store raw MAE for comparison
valid = td_mask_n2 & nnls_mask_n2 & gt_mask
raw_mae = {
    "NNLS": [compute_mae(nnls_abund_n2[i], gt_abundances[i], valid)
             for i in range(2)],
    "T-domain": [compute_mae(td_abund_n2[i], gt_abundances[i], valid)
                 for i in range(2)],
}

# Plot MAE vs TV weight
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for method, color, ls in [("NNLS", "#1f77b4", "--"), ("T-domain", "#2ca02c", "-")]:
    for i, iso in enumerate(isotope_names):
        maes = [tv_results[(method, w)][1 + i] for w in tv_weights]
        axes[i].plot(tv_weights, maes, f"{ls}", color=color,
                     marker="o", label=f"{method}+TV")
        axes[i].axhline(raw_mae[method][i], color=color, ls=":",
                        lw=0.8, label=f"{method} raw")
    axes[i].set_xlabel("TV weight")
    axes[i].set_ylabel("MAE vs ground truth")

for i, iso in enumerate(isotope_names):
    axes[i].set_title(f"{iso}")
    axes[i].set_xscale("log")
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

fig.suptitle(f"1b: TV Weight Sweep (n_incident={n_inc})", y=1.02)
fig.tight_layout()
save_fig(fig, "NB1_01b_tv_sweep")

# Find best weights
print(f"\n=== Best TV Weights (n={n_inc}) ===")
for method in ["NNLS", "T-domain"]:
    avg_maes = [(w, (tv_results[(method, w)][1] + tv_results[(method, w)][2]) / 2)
                for w in tv_weights]
    best_w, best_mae = min(avg_maes, key=lambda x: x[1])
    raw_avg = (raw_mae[method][0] + raw_mae[method][1]) / 2
    print(f"  {method}: best weight={best_w}, "
          f"MAE={best_mae:.4f} (raw={raw_avg:.4f}, "
          f"improvement={100*(raw_avg-best_mae)/raw_avg:.1f}%)")


In [ ]:
# --- 1b: Visual comparison at n=2 ---
# Show: GT / NNLS raw / NNLS+TV / T-domain raw / T-domain+TV

# Use best TV weights from sweep (pick weight with lowest avg MAE)
n_inc = 2

def find_best_tv(method):
    avg_maes = [(w, (tv_results[(method, w)][1] + tv_results[(method, w)][2]) / 2)
                for w in tv_weights]
    return min(avg_maes, key=lambda x: x[1])[0]

best_w_nnls = find_best_tv("NNLS")
best_w_td = find_best_tv("T-domain")
print(f"Best TV weights: NNLS={best_w_nnls}, T-domain={best_w_td}")

nnls_tv = tv_results[("NNLS", best_w_nnls)][0]
td_tv = tv_results[("T-domain", best_w_td)][0]

# Store for grand comparison
all_results[f"nnls_tv_best_n{n_inc}"] = nnls_tv
all_results[f"tdomain_tv_best_n{n_inc}"] = td_tv
all_results["best_w_nnls"] = best_w_nnls
all_results["best_w_td"] = best_w_td

methods = [
    ("Ground Truth", gt_abundances, gt_mask),
    (f"NNLS raw", nnls_abund_n2, nnls_mask_n2),
    (f"NNLS+TV(w={best_w_nnls})", nnls_tv, nnls_mask_n2),
    (f"T-domain raw", td_abund_n2, td_mask_n2),
    (f"T-domain+TV(w={best_w_td})", td_tv, td_mask_n2),
]

fig, axes = plt.subplots(2, len(methods), figsize=(4 * len(methods), 8))
for row_i, iso in enumerate(isotope_names):
    for col_j, (lbl, abund, mask) in enumerate(methods):
        amap = abund[row_i].copy()
        amap[~mask] = np.nan
        im = axes[row_i, col_j].imshow(amap, cmap="viridis",
                                        origin="upper", vmin=0, vmax=1)
        if row_i == 0:
            axes[row_i, col_j].set_title(lbl, fontsize=9)
        if col_j == 0:
            axes[row_i, col_j].set_ylabel(iso)
        fig.colorbar(im, ax=axes[row_i, col_j], shrink=0.7)

fig.suptitle(f"1b: Comparison at n_incident={n_inc}", y=1.02)
fig.tight_layout()
save_fig(fig, f"NB1_01b_comparison_n{n_inc}")

# Print MAE table
print(f"\n{'Method':<28} {'U-235 MAE':>12} {'Pu-241 MAE':>12} {'Avg MAE':>10}")
print("-" * 64)
for lbl, abund, mask in methods:
    valid = mask & gt_mask
    maes = [compute_mae(abund[i], gt_abundances[i], valid)
            for i in range(2)]
    print(f"{lbl:<28} {maes[0]:>12.4f} {maes[1]:>12.4f} "
          f"{np.mean(maes):>10.4f}")


### Experiment 1b: Observations

1. **Optimal TV weight**: What weight works best for each method?
   - **Both NNLS and T-domain: best weight = 0.1.** The TV sweep curves show a clear optimum
     around w=0.05-0.1. Below 0.02, TV doesn't smooth enough; above 0.2, over-smoothing
     begins (MAE rises). The U-235 and Pu-241 curves agree on the optimal weight, suggesting
     the same TV strength is appropriate for both isotope maps.

2. **Relative benefit**: Does TV help T-domain more or less than NNLS?
   - **TV helps T-domain more, both in absolute and relative terms.** T-domain+TV achieves
     MAE 0.108 (23.8% improvement over raw T-domain 0.141). NNLS+TV achieves MAE 0.131
     (15.1% improvement over raw NNLS 0.154). The T-domain maps have less systematic bias
     to begin with, so TV's spatial smoothing is more effective at reducing the remaining
     random noise. NNLS maps have both systematic bias and random noise; TV only helps with
     the latter.

3. **Over-smoothing**: At weight=0.5, are the logo edges blurred?
   - **Yes, moderately.** The TV sweep plot shows MAE begins rising for T-domain at w=0.5
     (U-235 curve turns up). The visual comparison confirms: at the optimal w=0.1, logo
     boundaries are somewhat softened but still clearly delineated. At w=0.5, the fine
     structure within the logos (e.g., the ORNL oak leaf veins, LANL atom orbitals) would
     be significantly blurred.

4. **Best combination so far**: Which (method + TV weight) gives lowest MAE?
   - **T-domain + TV (w=0.1): MAE 0.1075.** This is the clear winner at n=2, beating NNLS
     raw (0.154) by 30%, NNLS+TV (0.131) by 18%, and T-domain raw (0.141) by 24%. The
     visual comparison at n=2 shows a striking progression: NNLS raw is nearly featureless
     noise, while T-domain+TV recovers the Pu-241 logo shape and a hint of the U-235 logo.

---

# Experiment 1c: Iterative Alternating Optimization

**Question**: Can we improve further by alternating between T-domain fitting and
TV smoothing? The idea: TV-smoothed maps provide better initial guesses for the
next T-domain fit, which produces cleaner maps for the next TV step.

**Method**:
```
c^(0) = per-pixel T-domain fit (from Experiment 1a)
For k = 1..K:
  c_smooth^(k) = TV_denoise(c^(k-1), weight=lambda)
  c^(k) = per-pixel T-domain fit with warm start from c_smooth^(k)
  if MAE converged: break
```

**Key questions**:
1. Does the iterative scheme converge?
2. How many iterations does it take?
3. Does it improve over single-pass T-domain + TV?


In [ ]:
# --- 1c: Iterative alternating T-domain + TV ---

def iterative_tdomain_tv(data_cube, sigma_cube, A_matrix,
                         tv_weight, max_iter=5, gt=None, gt_mask=None,
                         label=""):
    """Alternating T-domain fit + TV smoothing.

    Parameters
    ----------
    data_cube, sigma_cube, A_matrix : as in fit_image_tdomain
    tv_weight : TV regularization weight
    max_iter : maximum iterations
    gt, gt_mask : ground truth for MAE tracking (optional)

    Returns
    -------
    final_abundances : (n_isotopes, H, W)
    final_mask : (H, W) bool
    history : list of dicts with per-iteration metrics
    """
    history = []

    # Iteration 0: initial T-domain fit (no warm start)
    print(f"  [{label}] Iter 0: initial T-domain fit...")
    coeffs, cost_map, mask = fit_image_tdomain(
        data_cube, sigma_cube, A_matrix, label=f"{label} iter0")
    abund = coeffs_to_abundances(coeffs, mask)

    if gt is not None:
        valid = mask & gt_mask
        maes = [compute_mae(abund[i], gt[i], valid)
                for i in range(abund.shape[0])]
        history.append({"iter": 0, "mae_u235": maes[0], "mae_pu241": maes[1],
                        "avg_mae": np.mean(maes)})
        print(f"    MAE: {maes[0]:.4f} / {maes[1]:.4f} (avg {np.mean(maes):.4f})")

    for k in range(1, max_iter + 1):
        print(f"  [{label}] Iter {k}: TV smoothing (w={tv_weight})...")
        smoothed = apply_tv(abund, tv_weight, mask)

        # Convert smoothed abundances back to coefficient scale for warm start
        # (undo normalization: use raw coefficients scaled by smoothed ratios)
        total_raw = np.nansum(coeffs, axis=0)
        x0_map = np.empty_like(coeffs)
        for i in range(coeffs.shape[0]):
            x0_map[i] = smoothed[i] * total_raw
        x0_map = np.maximum(x0_map, 0.01)  # avoid zero initial guess
        x0_map[:, ~mask] = 0.1  # default for failed pixels

        print(f"  [{label}] Iter {k}: T-domain refit with warm start...")
        coeffs, cost_map, mask = fit_image_tdomain(
            data_cube, sigma_cube, A_matrix, x0_map=x0_map,
            label=f"{label} iter{k}")
        abund = coeffs_to_abundances(coeffs, mask)

        if gt is not None:
            valid = mask & gt_mask
            maes = [compute_mae(abund[i], gt[i], valid)
                    for i in range(abund.shape[0])]
            history.append({"iter": k, "mae_u235": maes[0],
                            "mae_pu241": maes[1],
                            "avg_mae": np.mean(maes)})
            print(f"    MAE: {maes[0]:.4f} / {maes[1]:.4f} "
                  f"(avg {np.mean(maes):.4f})")

            # Check convergence
            if len(history) >= 2:
                prev = history[-2]["avg_mae"]
                curr = history[-1]["avg_mae"]
                improvement = (prev - curr) / prev if prev > 0 else 0
                print(f"    Improvement: {100*improvement:.2f}%")
                if improvement < 0.01:  # less than 1% improvement
                    print(f"    Converged at iteration {k}")
                    break

    # Final TV pass on the last coefficients
    final_abund = apply_tv(abund, tv_weight, mask)
    if gt is not None:
        valid = mask & gt_mask
        maes = [compute_mae(final_abund[i], gt[i], valid)
                for i in range(final_abund.shape[0])]
        history.append({"iter": k + 0.5, "mae_u235": maes[0],
                        "mae_pu241": maes[1],
                        "avg_mae": np.mean(maes), "note": "final TV"})
        print(f"  [{label}] Final TV: MAE {maes[0]:.4f} / {maes[1]:.4f}")

    return final_abund, mask, history


print("Iterative alternating function defined.")


In [ ]:
# --- 1c: Run iterative alternating at n=2 and n=10 ---
iter_results = {}

for n_inc in [2, 10]:
    print(f"\n{'='*60}")
    print(f"Iterative T-domain + TV at n_incident={n_inc}")
    print(f"{'='*60}")

    T_data = noisy_data[n_inc]
    sig = sigma_data[n_inc]
    tv_w = all_results.get("best_w_td", 0.1)

    final_abund, final_mask, history = iterative_tdomain_tv(
        T_data, sig, A_matrix,
        tv_weight=tv_w, max_iter=5,
        gt=gt_abundances, gt_mask=gt_mask,
        label=f"n={n_inc}")

    iter_results[n_inc] = {
        "abundances": final_abund,
        "mask": final_mask,
        "history": history,
    }
    all_results[f"tdomain_iter_n{n_inc}"] = final_abund
    all_results[f"tdomain_iter_n{n_inc}_mask"] = final_mask

# Convergence plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for idx, n_inc in enumerate([2, 10]):
    ax = axes[idx]
    hist = iter_results[n_inc]["history"]
    iters = [h["iter"] for h in hist]
    mae_u = [h["mae_u235"] for h in hist]
    mae_pu = [h["mae_pu241"] for h in hist]
    avg = [h["avg_mae"] for h in hist]
    ax.plot(iters, mae_u, "o-", label="U-235 MAE")
    ax.plot(iters, mae_pu, "s-", label="Pu-241 MAE")
    ax.plot(iters, avg, "^-", color="black", label="Average MAE")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("MAE vs ground truth")
    ax.set_title(f"n_incident={n_inc}")
    ax.legend()
    ax.grid(True, alpha=0.3)
fig.suptitle("1c: Iterative Alternating Convergence", y=1.02)
fig.tight_layout()
save_fig(fig, "NB1_01c_convergence")

# Abundance maps at n=2
n_inc = 2
fig = plot_abundance_maps(
    iter_results[n_inc]["abundances"], isotope_names,
    f"1c: Iterative T-domain+TV (n={n_inc})",
    iter_results[n_inc]["mask"])
save_fig(fig, f"NB1_01c_maps_n{n_inc}")


### Experiment 1c: Observations

1. **Convergence**: How many iterations does it take to converge?
   - **Just 1 iteration, effectively instant.** At both n=2 and n=10, the refit after TV
     smoothing produces MAE nearly identical to the initial fit (improvement < 0.03%).
     The convergence criterion (< 1% improvement) triggers immediately. The warm start
     from TV-smoothed coefficients does not help — the optimizer converges back to
     essentially the same solution regardless of initial guess.

2. **Improvement over single-pass**: Does iterating improve MAE beyond 1b?
   - **No.** The iterative scheme's final MAE (0.1075 at n=2, 0.0619 at n=10) is identical
     to single-pass T-domain+TV. The big MAE drop visible in the convergence plot (from
     0.141 to 0.108 at n=2) comes entirely from the final TV smoothing pass — NOT from
     the iterative re-fitting. The T-domain optimizer finds the same per-pixel minimum
     regardless of starting point, so warm-starting from TV-smoothed maps provides no
     benefit.

3. **Computational cost**: Is the extra time (multiple full-image fits) worth it?
   - **No.** Each T-domain full-image fit takes ~75-120s. The iterative scheme runs 2 fits
     (initial + 1 refit) before converging, costing ~200s total — versus ~75s for a single
     fit plus negligible TV time. For zero additional accuracy, this is pure waste.

4. **Visual quality**: Do the iterative maps look cleaner than single-pass?
   - **Identical.** The iterative T-domain+TV maps at n=2 are visually indistinguishable
     from single-pass T-domain+TV. Both show the same degree of spatial smoothing and the
     same logo visibility.

**Verdict**: The iterative alternating scheme is unnecessary. Single-pass T-domain + TV
achieves the same result at half the computational cost. The per-pixel T-domain optimizer
is robust enough that warm-starting from spatially-smoothed guesses does not change the
solution. This makes sense: `least_squares` with trust-region reflective is a global
method for 2-parameter problems — there are no local minima to escape.

---

# Grand Comparison

Side-by-side visual and quantitative comparison of all methods:

1. **NNLS raw** (attenuation domain, no regularization)
2. **NNLS + TV** (best TV weight from sweep)
3. **T-domain raw** (transmission domain, no regularization)
4. **T-domain + TV** (single-pass, best weight)
5. **T-domain + TV (iterative)** (alternating optimization)


In [ ]:
# --- Grand comparison at n=2 and n=10 ---

for n_inc in [2, 10]:
    print(f"\n=== Grand Comparison at n_incident={n_inc} ===")

    # Collect methods
    methods_list = [
        ("Ground Truth", gt_abundances, gt_mask),
        ("NNLS raw", all_results.get(f"nnls_n{n_inc}"),
         all_results.get(f"nnls_n{n_inc}_mask")),
    ]
    # Add NNLS+TV if available
    key_nnls_tv = f"nnls_tv_best_n{n_inc}"
    if key_nnls_tv in all_results:
        w = all_results.get("best_w_nnls", "?")
        methods_list.append(
            (f"NNLS+TV(w={w})", all_results[key_nnls_tv],
             all_results.get(f"nnls_n{n_inc}_mask")))

    methods_list.append(
        ("T-domain raw", all_results.get(f"tdomain_n{n_inc}"),
         all_results.get(f"tdomain_n{n_inc}_mask")))

    key_td_tv = f"tdomain_tv_best_n{n_inc}"
    if key_td_tv in all_results:
        w = all_results.get("best_w_td", "?")
        methods_list.append(
            (f"T-dom+TV(w={w})", all_results[key_td_tv],
             all_results.get(f"tdomain_n{n_inc}_mask")))

    key_iter = f"tdomain_iter_n{n_inc}"
    if key_iter in all_results:
        methods_list.append(
            ("T-dom+TV(iter)", all_results[key_iter],
             all_results.get(f"tdomain_iter_n{n_inc}_mask")))

    # Filter out missing
    methods_list = [(n, a, m) for n, a, m in methods_list
                    if a is not None and m is not None]

    n_methods = len(methods_list)
    fig, axes = plt.subplots(2, n_methods,
                              figsize=(3.5 * n_methods, 7))
    for row_i, iso in enumerate(isotope_names):
        for col_j, (lbl, abund, mask) in enumerate(methods_list):
            amap = abund[row_i].copy()
            amap[~mask] = np.nan
            im = axes[row_i, col_j].imshow(
                amap, cmap="viridis", origin="upper", vmin=0, vmax=1)
            if row_i == 0:
                axes[row_i, col_j].set_title(lbl, fontsize=8)
            if col_j == 0:
                axes[row_i, col_j].set_ylabel(iso)
            fig.colorbar(im, ax=axes[row_i, col_j], shrink=0.6)
    fig.suptitle(f"Grand Comparison at n_incident={n_inc}", y=1.02)
    fig.tight_layout()
    save_fig(fig, f"NB1_02_grand_comparison_n{n_inc}")


In [ ]:
# --- Summary metrics table ---
import pandas as pd

summary_rows = []

for n_inc in noise_levels:
    T_data = noisy_data[n_inc]
    sig = sigma_data[n_inc]

    # For noise levels not already computed, do quick per-pixel fits
    nnls_key = f"nnls_n{n_inc}"
    td_key = f"tdomain_n{n_inc}"

    if nnls_key not in all_results:
        print(f"Computing NNLS for n={n_inc}...")
        c_nnls, m_nnls = fit_image_nnls(T_data, sig, A_matrix,
                                         label=f"n={n_inc}")
        a_nnls = coeffs_to_abundances(c_nnls, m_nnls)
        all_results[nnls_key] = a_nnls
        all_results[f"{nnls_key}_mask"] = m_nnls

    if td_key not in all_results:
        print(f"Computing T-domain for n={n_inc}...")
        c_td, _, m_td = fit_image_tdomain(T_data, sig, A_matrix,
                                           label=f"n={n_inc}")
        a_td = coeffs_to_abundances(c_td, m_td)
        all_results[td_key] = a_td
        all_results[f"{td_key}_mask"] = m_td

    # Compute metrics for all available methods
    methods = [
        ("NNLS", all_results[nnls_key], all_results[f"{nnls_key}_mask"]),
        ("T-domain", all_results[td_key], all_results[f"{td_key}_mask"]),
    ]

    # TV versions (only for n=2 where we computed them)
    if f"nnls_tv_best_n{n_inc}" in all_results:
        methods.append(("NNLS+TV", all_results[f"nnls_tv_best_n{n_inc}"],
                        all_results[f"{nnls_key}_mask"]))
    if f"tdomain_tv_best_n{n_inc}" in all_results:
        methods.append(("T-dom+TV", all_results[f"tdomain_tv_best_n{n_inc}"],
                        all_results[f"{td_key}_mask"]))
    if f"tdomain_iter_n{n_inc}" in all_results:
        methods.append(("T-dom+TV(iter)",
                        all_results[f"tdomain_iter_n{n_inc}"],
                        all_results[f"tdomain_iter_n{n_inc}_mask"]))

    for method_name, abund, mask in methods:
        valid = mask & gt_mask
        for i, iso in enumerate(isotope_names):
            mae = compute_mae(abund[i], gt_abundances[i], valid)
            rmse = compute_rmse(abund[i], gt_abundances[i], valid)
            summary_rows.append({
                "n_incident": n_inc,
                "method": method_name,
                "isotope": iso,
                "MAE": mae,
                "RMSE": rmse,
            })

df_summary = pd.DataFrame(summary_rows)

# Pivot for display
for iso in isotope_names:
    print(f"\n=== {iso} ===")
    sub = df_summary[df_summary["isotope"] == iso]
    pivot = sub.pivot_table(index="method", columns="n_incident",
                            values="MAE", sort=False)
    pivot = pivot.reindex(columns=sorted(pivot.columns))
    print(pivot.to_string(float_format=lambda x: f"{x:.4f}"))

# Visual summary: MAE bar chart at n=2
n_inc = 2
sub = df_summary[df_summary["n_incident"] == n_inc]
methods_order = sub["method"].unique()
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(methods_order))
w_bar = 0.35
for i, iso in enumerate(isotope_names):
    vals = [sub[(sub["method"] == m) & (sub["isotope"] == iso)]["MAE"].values[0]
            for m in methods_order]
    ax.bar(x + i * w_bar, vals, w_bar, label=iso)
ax.set_xticks(x + w_bar / 2)
ax.set_xticklabels(methods_order, rotation=30, ha="right")
ax.set_ylabel("MAE vs ground truth")
ax.set_title(f"Summary: MAE at n_incident={n_inc}")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
save_fig(fig, "NB1_03_summary_metrics")


---

# Final Observations

## Summary of Results

| Method | n=2 MAE | n=5 MAE | n=10 MAE | n=50 MAE | n=500 MAE |
|--------|---------|---------|----------|----------|-----------|
| NNLS raw | 0.154 | 0.124 | 0.102 | 0.054 | **0.013** |
| T-domain raw | 0.141 | 0.116 | 0.082 | **0.031** | 0.029 |
| NNLS + TV (w=0.1) | 0.131 | — | — | — | — |
| T-domain + TV (w=0.1) | **0.108** | — | — | — | — |
| T-domain + TV (iter) | **0.108** | — | 0.062 | — | — |

## Key Findings

1. **Does T-domain avoid the clipping catastrophe?**
   - **Partially.** T-domain avoids the log-clipping that converts T=0 into A=13.8 outliers.
     This gives a consistent advantage at moderate-to-heavy noise (n=5-50), with the biggest
     win at n=10 (20% MAE reduction) and n=50 (43% reduction). However, at extreme noise
     (n=2), the improvement is only 8% because the fundamental problem — 2 photons per bin
     carry almost no information — cannot be solved by changing the fitting domain alone.
     **Unexpected finding:** T-domain has a systematic bias vs NNLS even on clean data
     (MAE 0.036), because the two methods weight energy bins differently. At very low noise
     (n=500), this bias makes T-domain *worse* than NNLS.

2. **How much does TV regularization help?**
   - **Substantially.** TV provides the single largest improvement at n=2: from 0.141 to
     0.108 (24% reduction on top of T-domain). Optimal weight is 0.1 for both methods.
     TV helps T-domain more than NNLS because T-domain maps have less systematic bias,
     leaving mostly random noise that TV can smooth. The combination T-domain+TV achieves
     30% lower MAE than raw NNLS (0.108 vs 0.154).

3. **Is iterative alternating worth the computational cost?**
   - **No.** The iterative scheme converges in 1 iteration with <0.03% improvement. The
     per-pixel optimizer finds the same solution regardless of initial guess — there are
     no local minima in this 2-parameter problem. All improvement comes from the final
     TV pass, not from re-fitting. Recommendation: use single-pass T-domain + TV.

4. **What is the best method for n=2 (extreme noise)?**
   - **T-domain + TV (w=0.1), MAE = 0.108.** This is the best result we've achieved, but
     it is still far from clean: average per-pixel abundance error of ~11 percentage points.
     The logos are visible but blurry. For comparison, the ground truth has U-235 abundances
     ranging 0.26-0.92, so an MAE of 0.11 is roughly a 15-20% relative error.

5. **What is the best method for n=10 (moderate noise)?**
   - **T-domain + TV, MAE = 0.062.** At this noise level, the logos are clearly recovered
     with recognizable detail. The T-domain advantage over NNLS is more pronounced (0.062
     vs 0.102 raw NNLS), and the maps are visually close to the ground truth. This is the
     regime where T-domain fitting shows its greatest practical benefit.

## Implications for Notebooks 2-4

- **Baseline to beat**: T-domain+TV at n=2 achieves MAE 0.108. Any NMF/Bayesian/BM3D
  approach must beat this to be worth pursuing.
- **The core unsolved problem**: At n=2, per-pixel information is fundamentally inadequate.
  Methods that borrow strength across *multiple pixels simultaneously* (not just post-hoc
  smoothing) may be able to do better. This motivates:
  - **NB2 (Over-complete NMF)**: Uses spatial redundancy across all 65K pixels during
    decomposition, not just post-hoc smoothing.
  - **NB3 (Bayesian MAP)**: Jointly optimizes data fidelity + spatial prior in a single
    objective, potentially better than the sequential fit-then-smooth approach.
  - **NB4 (BM3D)**: Exploits non-local spatial self-similarity for spectral denoising.
- **T-domain's systematic bias** at low noise suggests that a method working in the
  attenuation domain but with proper outlier handling (e.g., robust regression, Huber loss)
  might combine the best of both worlds.

## Next Steps

- Notebook 2: Over-complete NMF + SAMMY-guided rank selection
- Notebook 3: Bayesian MAP with SAMMY forward model
- Notebook 4: BM3D-like spectral denoising (if time)